# TF-IDF Baseline

Our proposal also mentioned a TF-IDF baseline. This notebook runs the same resume-job comparison using TF-IDF cosine similarity instead of SBERT embeddings. We then check whether the demographic-sensitivity pattern we see with SBERT shows up with TF-IDF too.

TF-IDF is a much older, simpler representation. If it shows similar score shifts when we change a name, that tells us the effect is not unique to neural embeddings.

In [ ]:
!pip install scikit-learn pandas

In [ ]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded.")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs:", jobs.shape)
print("Resumes:", resumes.shape)

In [ ]:
def make_job_text(row):
    return f"{row['title']} {row['domain']} {row['company_name']} {row['job_description']}"

jobs["job_text"] = jobs.apply(make_job_text, axis=1)

In [ ]:
# Fit one TF-IDF model on every resume and job text.
all_texts = list(resumes["resume_text"]) + list(jobs["job_text"])
vec = TfidfVectorizer(stop_words="english", lowercase=True, min_df=1)
vec.fit(all_texts)

resume_vectors = vec.transform(resumes["resume_text"])
job_vectors = vec.transform(jobs["job_text"])
print("Resume vector shape:", resume_vectors.shape)
print("Job vector shape:", job_vectors.shape)

In [ ]:
rows = []
for i, resume_row in resumes.reset_index(drop=True).iterrows():
    for j, job_row in jobs.reset_index(drop=True).iterrows():
        score = float(cosine_similarity(resume_vectors[i], job_vectors[j])[0][0])
        rows.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "tfidf_similarity_score": score,
        })

tfidf_scores = pd.DataFrame(rows)
print("TF-IDF scores:", len(tfidf_scores))
display(tfidf_scores.head())

In [ ]:
# Fairness check: compare each counterfactual TF-IDF score to its original.
orig = (
    tfidf_scores[tfidf_scores["version"] == "original"]
    [["resume_id", "job_id", "job_title", "tfidf_similarity_score"]]
    .rename(columns={"tfidf_similarity_score": "tfidf_original_score"})
)
changed = tfidf_scores[tfidf_scores["version"] != "original"].rename(
    columns={"tfidf_similarity_score": "tfidf_changed_score"}
)

tfidf_comparison = changed.merge(orig, on=["resume_id", "job_id", "job_title"], how="left")
tfidf_comparison["tfidf_score_difference"] = (
    tfidf_comparison["tfidf_changed_score"] - tfidf_comparison["tfidf_original_score"]
)
tfidf_comparison["tfidf_absolute_difference"] = tfidf_comparison["tfidf_score_difference"].abs()
display(tfidf_comparison.head())

In [ ]:
tfidf_summary = tfidf_comparison.groupby("changed_signal").agg(
    average_tfidf_score_difference=("tfidf_score_difference", "mean"),
    average_tfidf_absolute_difference=("tfidf_absolute_difference", "mean"),
    max_tfidf_absolute_difference=("tfidf_absolute_difference", "max"),
).reset_index()
display(tfidf_summary)

## How to read this

Compare these numbers with the SBERT fairness summary. If TF-IDF shows similar shifts on the same kind of demographic change, that means the bias signal we measured with SBERT is not specifically a neural-embedding problem; it is partly a property of the resume text itself. If TF-IDF shows much smaller shifts, then SBERT is adding extra sensitivity beyond what raw word overlap would predict.

In [ ]:
tfidf_scores.to_csv("results/tfidf_scores.csv", index=False)
tfidf_comparison.to_csv("results/tfidf_comparison.csv", index=False)
tfidf_summary.to_csv("results/tfidf_summary.csv", index=False)
print("Saved.")

In [ ]:
from google.colab import files
files.download("results/tfidf_scores.csv")
files.download("results/tfidf_comparison.csv")
files.download("results/tfidf_summary.csv")